# Field Shift · Punto 5: condados gemelos

**Qué ve el productor:** en la pantalla "Próxima campaña", al lado de cada opción, de 3 a 5 condados con suelo y clima parecidos al suyo que **ya hicieron** esa opción, y cómo les fue. Por ejemplo:

> "Counties like yours that grow winter wheat: 4. Similar soil and weather doesn't guarantee the same result."

**No entrena ningún modelo.** Es una búsqueda de vecinos (kNN, *k nearest neighbors*): se mide qué tan distinto es cada condado de cada otro y se quedan los más cercanos. No es el modelo A ni el B del `CLAUDE.md`: no predice rinde.

**Se arma en dos partes:**

- **Parte 1 (pasos 0 a 10): encontrar los parecidos.** Solo con suelo y clima, que el productor no elige. La rotación y el rinde **no** entran al parecido: son justamente lo que después se mira en los gemelos. El paso 9 controla si el parecido dice algo más que el mapa.
- **Parte 2 (pasos 11 a 20): los gemelos de cada opción.** De esos parecidos, quedarse con los que hicieron cada opción (maíz o soja rotados o continuos, trigo de invierno, avena), armar con sus rindes un rango p10 / mediana / p90, compararlo con el modelo del punto 2, controlar que le ganen a los vecinos del mapa (paso 19) y guardar el JSON con el formato del front (`web/src/next/types.ts`).

**Cómo usarla:** correr todo de arriba hacia abajo. Usa `backend/Datos/tabla_modelado.csv`, que deja el punto 2, y la entrega `backend/Datos/entregas/fieldshift_datos_v1.3/` (nombres, centroides, lluvia IMERG, trigo y avena).

## 0. Librerías

- **pandas / numpy:** tablas y cuentas.
- **scikit-learn:** de acá sale `nan_euclidean_distances`, que mide distancias aunque a un condado le falte algún dato.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import nan_euclidean_distances

## 1. Parámetros

Todo lo que se puede cambiar está en esta celda.

- `VERSION_ENTREGA`: la entrega vigente es la v1.3. Su carpeta `condado/` es idéntica a la de la v1.2 (solo cambia `lotes/`), así que los resultados no cambian.
- `VARIABLES_SUELO` y `VARIABLES_CLIMA`: con qué se mide el parecido. Se eligieron para que se pueda explicar en una frase ("suelo y clima parecidos") y sin pares casi repetidos: `twi_medio` queda afuera porque se mueve igual que la pendiente (correlación 0,94), y `deficit_temporada_mm` porque se mueve casi igual que la lluvia de la temporada (0,85).
- `LLUVIA`: de dónde sale la lluvia.
  - `"imerg"` (la que se usa): lluvia GPM IMERG de la entrega, el **promedio del condado** a 0,1° (unos 10 km).
  - `"power"`: lluvia de NASA POWER, que es el valor del **centroide** en una grilla de 0,5° (unos 50 km). Muchos condados vecinos caen en la misma celda y tienen **exactamente** la misma lluvia todos los años (el paso 2 los cuenta). Así la búsqueda no los puede distinguir y termina eligiendo a los vecinos del mapa.
  - Con IMERG los gemelos le ganan a los vecinos del mapa en las 6 opciones del control del paso 19; con POWER pierden en 2 (avena y maíz sobre maíz). La temperatura sigue siendo de POWER: la entrega no trae otra.
- `PESO_SUELO`: cuánto pesa el suelo frente al clima. Con 0,5 pesan lo mismo, aunque uno tenga más variables que el otro.
- `N_VECINOS`: cuántos parecidos se guardan por condado. Se guardan más de los 3 a 5 que muestra el front porque la parte 2 va a descartar los que no hicieron la opción.
- `PERCENTIL_MAXIMO`: el filtro de "parecido de verdad". Un vecino cuenta solo si está más cerca que el 10 % de los pares de condados más parecidos de toda la región. Así, un condado raro puede quedar con menos de 3 gemelos, y el front muestra confianza baja en vez de inventar parecidos.
- `CULTIVOS_AREA`: los cultivos con superficie sembrada de NASS. Con ellos se calcula qué parte del área ocupa cada uno en un condado (`grew_crop_pct` del front).
- Solo los años 2009–2023. 2024–2025 son la prueba final del punto 2 y no se miran.

In [2]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
SALIDA = RAIZ / "backend" / "Datos"
RUTA_TABLA = SALIDA / "tabla_modelado.csv"
VERSION_ENTREGA = "todo_10estados_20261115"   # 10 estados (IA, IL, IN, KS, MN, MO, NE, OH, SD, WI) con las columnas de la v3
RUTA_ENTREGA = RAIZ / "backend" / "Datos" / "entregas" / f"fieldshift_datos_{VERSION_ENTREGA}" / "unido_10_estados"   # nombres, centroides, IMERG, trigo y avena

ANIOS_PRUEBA = [2024, 2025]
NOMBRES_ESTADO = {"17": "Illinois", "18": "Indiana", "19": "Iowa", "20": "Kansas", "27": "Minnesota",
                  "29": "Missouri", "31": "Nebraska", "39": "Ohio", "46": "South Dakota", "55": "Wisconsin"}
SIGLAS_ESTADO = {"17": "IL", "18": "IN", "19": "IA", "20": "KS", "27": "MN", "29": "MO", "31": "NE",
                 "39": "OH", "46": "SD", "55": "WI"}   # el front usa la sigla ("Story, IA")

# Suelo: fijo, no cambia de un año a otro (SSURGO)
VARIABLES_SUELO = ["aws0_100_mm", "nccpi_maiz", "soc0_30_gm2", "pendiente_media_pct", "clase_drenaje"]
# Clima: se resume con el año típico del condado. El calor es de NASA POWER; la lluvia, según LLUVIA
LLUVIA = "imerg"   # "imerg" (promedio del condado, ~10 km) o "power" (centroide, ~50 km)
CLIMA_SEGUN_LLUVIA = {
    "imerg": ["prec_temporada_imerg_mm", "gdd_temporada", "tmax_jul_c", "prec_imerg_mm_05"],
    "power": ["prec_temporada_mm", "gdd_temporada", "tmax_jul_c", "prec_power_mm_05"],
}
VARIABLES_CLIMA = CLIMA_SEGUN_LLUVIA[LLUVIA]

# Nombre en inglés para el front ("Similar on ...") y en español para revisar
ETIQUETAS = {
    "aws0_100_mm": ("water the soil can hold", "agua que retiene el suelo"),
    "nccpi_maiz": ("soil productivity", "productividad del suelo"),
    "soc0_30_gm2": ("soil organic carbon", "carbono del suelo"),
    "pendiente_media_pct": ("slope", "pendiente"),
    "clase_drenaje": ("soil drainage", "drenaje del suelo"),
    "prec_temporada_mm": ("growing-season rain", "lluvia de la temporada"),
    "prec_temporada_imerg_mm": ("growing-season rain", "lluvia de la temporada"),
    "gdd_temporada": ("growing-season heat", "calor de la temporada"),
    "tmax_jul_c": ("July heat", "calor de julio"),
    "prec_power_mm_05": ("May rain", "lluvia de mayo"),
    "prec_imerg_mm_05": ("May rain", "lluvia de mayo"),
}

# El drenaje es texto: se pasa a un número ordenado (1 = drena muy mal, 7 = drena muy rápido)
ORDEN_DRENAJE = {
    "Very poorly drained": 1,
    "Poorly drained": 2,
    "Somewhat poorly drained": 3,
    "Moderately well drained": 4,
    "Well drained": 5,
    "Somewhat excessively drained": 6,
    "Excessively drained": 7,
}

PESO_SUELO = 0.5          # 0,5 = suelo y clima pesan lo mismo
N_VECINOS = 15            # parecidos que se guardan por condado
PERCENTIL_MAXIMO = 10     # un vecino cuenta si está entre el 10 % de pares más cercanos de la región
N_SIMILAR_ON = 2          # en cuántas variables se dice "Similar on ..."
CONDADOS_EJEMPLO = ["19169", "17113", "20191"]   # Story (IA), McLean (IL) y Sumner (KS), el del video
CULTIVOS_AREA = ["maiz", "soja", "trigo_invierno", "avena"]   # superficie sembrada de NASS, para "% del área"

# De dónde salen suelo, clima, rinde y rotación de cada condado:
#   "entrega": directo de la entrega (los 9 estados, sin esperar al punto 2)
#   "punto2": backend/Datos/tabla_modelado.csv (solo los estados con los que corrió el punto 2)
FUENTE_TABLA = "entrega"
ANIO_DESDE = 2009   # desde ese año el CDL trae la rotación de cada condado (igual que el punto 2)
COLUMNAS_ENTREGA = ["fips", "anio", "cultivo", "rend_bu_ac", "aws0_100_mm", "nccpi_maiz", "soc0_30_gm2",
                    "pendiente_media_pct", "clase_drenaje", "prec_temporada_mm", "gdd_temporada", "tmax_jul_c",
                    "pct_maiz_tras_maiz", "pct_soja_tras_soja"]
HAY_DATOS = RUTA_TABLA.exists() or (RUTA_ENTREGA / "tabla_anual.csv").exists()
HAY_ENTREGA = (RUTA_ENTREGA / "tabla_anual.csv").exists() and (RUTA_ENTREGA / "referencia" / "condados.csv").exists()
print("¿Está la tabla del punto 2?", HAY_DATOS, f"| ¿Está la entrega {VERSION_ENTREGA}?", HAY_ENTREGA)
if not HAY_DATOS:
    print("Falta backend/Datos/tabla_modelado.csv: correr punto2_rotaciones.ipynb. La notebook queda en espera.")
if not HAY_ENTREGA:
    print(f"Falta la entrega en backend/Datos/entregas/fieldshift_datos_{VERSION_ENTREGA}/: sin nombres, centroides, IMERG, trigo ni avena.")
    if LLUVIA == "imerg":
        LLUVIA = "power"
        VARIABLES_CLIMA = CLIMA_SEGUN_LLUVIA[LLUVIA]
        print("Sin la entrega no hay lluvia IMERG: se usa la de POWER (los gemelos van a parecerse más a los vecinos del mapa).")

¿Está la tabla del punto 2? True | ¿Está la entrega todo_10estados_20261115? True


## 2. Cargar la tabla

`tabla_modelado.csv` tiene una fila por condado, año y cultivo (maíz o soja). El suelo se repite en todas las filas de un condado; el clima cambia cada año.

De la entrega se cargan también:

- **Nombre y centroide** (latitud y longitud del centro) de cada condado, de `referencia/condados.csv`. El front los usa para escribir el nombre del gemelo y dibujar el arco en el mapa. Con los centroides se calcula también la **distancia en el mapa** entre cada par de condados (en km), para los controles de los pasos 9 y 19.
- **La lluvia IMERG**, de `tabla_mensual.csv`: la suma de abril a septiembre (temporada) y la de mayo, las mismas dos lluvias que se usaban de POWER. Se toman **todos** los años 2009–2023, aunque al condado le falte el rinde de alguno: el clima existe igual, y con más años la mediana es más estable.
- **Qué parte del área ocupa cada cultivo**, de la superficie sembrada de NASS en `tabla_anual.csv`: la mediana 2009–2023 del área de cada cultivo sobre el total de maíz, soja, trigo y avena. Es lo que el front muestra como "Corn on 48% of fields".

Si una variable pedida no viene, se avisa y se sigue sin ella. No se inventa.

Al final se cuenta cuántos condados tienen **la misma lluvia exacta** que otro condado todos los años, con POWER y con IMERG. Es el motivo de usar IMERG (paso 1).

In [3]:
if HAY_DATOS:
    if FUENTE_TABLA == "entrega" and (RUTA_ENTREGA / "tabla_anual.csv").exists():
        # Misma forma que tabla_modelado.csv: maíz y soja con rinde, una fila por condado, año y cultivo
        tabla = pd.read_csv(RUTA_ENTREGA / "tabla_anual.csv", dtype={"fips": str}, usecols=COLUMNAS_ENTREGA)
        tabla = tabla[tabla["cultivo"].isin(["maiz", "soja"]) & tabla["rend_bu_ac"].notna() & (tabla["anio"] >= ANIO_DESDE)]
        mayo = pd.read_csv(RUTA_ENTREGA / "tabla_mensual.csv", dtype={"fips": str}, usecols=["fips", "anio", "mes", "prec_power_mm"])
        mayo = mayo[mayo["mes"].str[5:7] == "05"].rename(columns={"prec_power_mm": "prec_power_mm_05"}).drop(columns="mes")
        tabla = tabla.merge(mayo, on=["fips", "anio"], how="left")
        tabla.insert(2, "estado", tabla["fips"].str[:2])
        print("Suelo, clima, rinde y rotación: de la entrega", VERSION_ENTREGA)
    else:
        tabla = pd.read_csv(RUTA_TABLA, dtype={"fips": str, "estado": str})
        print("Suelo, clima, rinde y rotación: de backend/Datos/tabla_modelado.csv (punto 2)")
    tabla = tabla[~tabla["anio"].isin(ANIOS_PRUEBA)].copy()
    ANIOS = (tabla["anio"].min(), tabla["anio"].max())
else:
    ANIOS = (2009, 2023)
    print("En espera de la tabla del punto 2.")

CONDADOS = pd.DataFrame(columns=["condado", "lat_centroide", "lon_centroide"])
PARTE_AREA = pd.DataFrame(columns=CULTIVOS_AREA)
KM = None
LLUVIA_IMERG = pd.DataFrame()
if HAY_ENTREGA:
    CONDADOS = pd.read_csv(RUTA_ENTREGA / "referencia" / "condados.csv", dtype={"fips": str}).set_index("fips")
    print(f"Nombres y centroides de {len(CONDADOS)} condados")

    # Distancia en el mapa entre centroides (km, fórmula del gran círculo)
    lat = np.radians(CONDADOS["lat_centroide"].to_numpy())
    lon = np.radians(CONDADOS["lon_centroide"].to_numpy())
    h = np.sin((lat[:, None] - lat) / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat) * np.sin((lon[:, None] - lon) / 2) ** 2
    KM = pd.DataFrame(6371 * 2 * np.arcsin(np.sqrt(h)), index=CONDADOS.index, columns=CONDADOS.index)

    # Superficie sembrada de NASS: qué parte del área ocupa cada cultivo (mediana de los años)
    anual = pd.read_csv(RUTA_ENTREGA / "tabla_anual.csv", dtype={"fips": str},
                        usecols=["fips", "anio", "cultivo", "rend_bu_ac", "sup_sembrada_ac"])
    anual = anual[anual["anio"].between(*ANIOS)].copy()
    anual["estado"] = anual["fips"].str[:2]
    area = anual[anual["cultivo"].isin(CULTIVOS_AREA)].pivot_table(index=["fips", "anio"], columns="cultivo", values="sup_sembrada_ac")
    PARTE_AREA = area.div(area.sum(axis=1, min_count=1), axis=0).mul(100).groupby("fips").median()

    if HAY_DATOS:
        # Lluvia IMERG: suma de abril a septiembre (solo si están los 6 meses) y la de mayo
        mensual = pd.read_csv(RUTA_ENTREGA / "tabla_mensual.csv", dtype={"fips": str}, usecols=["fips", "anio", "mes", "prec_imerg_mm"])
        mensual["m"] = mensual["mes"].str[5:7].astype(int)
        temporada = mensual[mensual["m"].between(4, 9)]
        LLUVIA_IMERG = pd.DataFrame({
            "prec_temporada_imerg_mm": temporada.groupby(["fips", "anio"])["prec_imerg_mm"].sum(min_count=6),
            "prec_imerg_mm_05": mensual[mensual["m"] == 5].set_index(["fips", "anio"])["prec_imerg_mm"],
        })
        LLUVIA_IMERG = LLUVIA_IMERG[LLUVIA_IMERG.index.get_level_values("anio").to_series().between(*ANIOS).to_numpy()]

if HAY_DATOS:
    hay = set(tabla.columns) | set(LLUVIA_IMERG.columns)
    faltan = [c for c in VARIABLES_SUELO + VARIABLES_CLIMA if c not in hay]
    if faltan:
        print("No vinieron en la tabla, se sigue sin ellas:", faltan)
    VARIABLES_SUELO = [c for c in VARIABLES_SUELO if c in hay]
    VARIABLES_CLIMA = [c for c in VARIABLES_CLIMA if c in hay]

    print(f"{tabla['fips'].nunique()} condados | años {ANIOS[0]}–{ANIOS[1]} | lluvia: {LLUVIA.upper()}")
    print("Condados por estado:", tabla.groupby("estado")["fips"].nunique().rename(NOMBRES_ESTADO).to_dict())

    # Por qué IMERG: condados con la misma lluvia de la temporada que otro condado, todos los años
    for columna, fuente, origen in [("prec_temporada_mm", "POWER", tabla), ("prec_temporada_imerg_mm", "IMERG", LLUVIA_IMERG.reset_index())]:
        if columna in origen.columns:
            serie = origen.groupby(["fips", "anio"])[columna].first().unstack().round(1)
            print(f"Condados con la misma lluvia de {fuente} que otro condado, todos los años: "
                  f"{serie.duplicated(keep=False).sum()} de {len(serie)}")


def nombrar(fips):
    """'Story (Iowa)' si está la entrega; si no, el fips."""
    nombre = CONDADOS["condado"].get(fips)
    return f"{nombre} ({NOMBRES_ESTADO.get(fips[:2], fips[:2])})" if isinstance(nombre, str) else fips

Suelo, clima, rinde y rotación: de la entrega todo_10estados_20261115
Nombres y centroides de 919 condados


895 condados | años 2009–2023 | lluvia: IMERG
Condados por estado: {'Illinois': 102, 'Indiana': 92, 'Iowa': 99, 'Kansas': 105, 'Minnesota': 84, 'Missouri': 104, 'Nebraska': 90, 'Ohio': 87, 'South Dakota': 66, 'Wisconsin': 66}
Condados con la misma lluvia de POWER que otro condado, todos los años: 240 de 895
Condados con la misma lluvia de IMERG que otro condado, todos los años: 0 de 919


## 3. El perfil de cada condado

Cada condado queda resumido en **una sola fila** con 9 números:

- **Suelo:** el valor del condado, que es el mismo todos los años. El drenaje se pasa de texto a número con `ORDEN_DRENAJE`.
- **Clima típico:** la **mediana** de 2009–2023. La mediana es el valor del medio: la mitad de los años estuvo arriba y la mitad abajo. Se usa en vez del promedio para que un año extremo, como la sequía de 2012, no cambie lo que se considera "el clima de ese condado".

Antes de seguir se chequea que el suelo realmente no cambie entre años: si cambiara, algo raro hay en la tabla.

In [4]:
if HAY_DATOS:
    tabla["clase_drenaje"] = tabla["clase_drenaje"].map(ORDEN_DRENAJE)

    cambia = tabla.groupby("fips")[VARIABLES_SUELO].nunique().gt(1).any()
    if cambia.any():
        print("Ojo: estas variables de suelo cambian entre años:", cambia[cambia].index.tolist())

    suelo = tabla.groupby("fips")[VARIABLES_SUELO].first()
    # Una fila por condado y año (la tabla repite el clima en maíz y soja), y después la mediana de los años
    del_punto2 = [c for c in VARIABLES_CLIMA if c in tabla.columns]
    de_imerg = [c for c in VARIABLES_CLIMA if c not in tabla.columns]
    clima = tabla.groupby(["fips", "anio"])[del_punto2].first().groupby("fips").median()
    if de_imerg:
        clima = clima.join(LLUVIA_IMERG[de_imerg].groupby("fips").median())
    clima = clima[VARIABLES_CLIMA]
    perfil = suelo.join(clima)
    perfil.insert(0, "estado", perfil.index.str[:2])

    print("Datos faltantes por variable:", perfil.isna().sum()[lambda s: s > 0].to_dict() or "ninguno")
    display(perfil.loc[[f for f in CONDADOS_EJEMPLO if f in perfil.index]])
    display(perfil.describe().T[["mean", "min", "50%", "max"]].round(1))
else:
    print("En espera de la tabla del punto 2.")

Datos faltantes por variable: {'aws0_100_mm': 9, 'nccpi_maiz': 9, 'soc0_30_gm2': 9, 'clase_drenaje': 9}


,estado,aws0_100_mm,nccpi_maiz,soc0_30_gm2,pendiente_media_pct,clase_drenaje,prec_temporada_imerg_mm,gdd_temporada,tmax_jul_c,prec_imerg_mm_05
fips,,,,,,,,,,
19169,19,182.6,0.7303,9419.0,2.439,2.0,677.080,1738.460,28.865,110.855
17113,17,172.4,0.7832,8195.0,1.593,2.0,640.468,1826.195,28.671,134.198
20191,20,166.6,0.4575,4238.0,1.829,5.0,633.613,2312.590,33.606,130.522


,mean,min,50%,max
aws0_100_mm,161.6,65.5,166.0,302.4
nccpi_maiz,0.5,0.1,0.6,0.9
soc0_30_gm2,6047.4,1788.0,5607.5,19178.0
pendiente_media_pct,3.6,1.0,2.9,14.8
clase_drenaje,4.1,1.0,5.0,7.0
prec_temporada_imerg_mm,609.3,325.0,632.7,774.0
gdd_temporada,1795.2,1044.0,1795.9,2324.8
tmax_jul_c,29.7,23.5,29.4,37.4
prec_imerg_mm_05,112.2,54.0,110.4,201.9


## 4. Poner todo en la misma escala

Las variables están en unidades muy distintas: el carbono del suelo anda por los 6.000 g/m² y la productividad del suelo entre 0 y 1. Si las comparamos así, el carbono taparía todo lo demás.

Por eso cada variable se pasa a **cuántos desvíos se aparta del promedio de la región** (*z-score*): 0 es un condado típico, +1 es "bastante más que lo típico" y −1, "bastante menos". Así, una diferencia de 1 significa lo mismo en cualquier variable.

Después se aplica el peso: el suelo tiene 5 variables y el clima 4. Para que pesen lo mismo, cada grupo se divide por la raíz de su cantidad de variables y se multiplica por su peso (`PESO_SUELO`).

In [5]:
if HAY_DATOS:
    COLUMNAS = VARIABLES_SUELO + VARIABLES_CLIMA
    z = (perfil[COLUMNAS] - perfil[COLUMNAS].mean()) / perfil[COLUMNAS].std()

    pesos = pd.Series(
        {c: np.sqrt(PESO_SUELO / len(VARIABLES_SUELO)) for c in VARIABLES_SUELO}
        | {c: np.sqrt((1 - PESO_SUELO) / len(VARIABLES_CLIMA)) for c in VARIABLES_CLIMA}
    )
    x = z * pesos
    display(z.loc[[f for f in CONDADOS_EJEMPLO if f in z.index]].round(2))
else:
    print("En espera de la tabla del punto 2.")

,aws0_100_mm,nccpi_maiz,soc0_30_gm2,pendiente_media_pct,clase_drenaje,prec_temporada_imerg_mm,gdd_temporada,tmax_jul_c,prec_imerg_mm_05
fips,,,,,,,,,
19169,0.77,1.03,1.40,-0.52,-1.69,0.74,-0.22,-0.40,-0.06
17113,0.40,1.34,0.89,-0.91,-1.69,0.34,0.12,-0.50,0.94
20191,0.19,-0.53,-0.75,-0.80,0.69,0.26,2.02,1.88,0.78


## 5. La distancia entre cada par de condados

Imaginá un mapa, pero con 9 ejes en vez de 2: uno por variable. Cada condado es un punto en ese mapa. La **distancia** entre dos condados es la línea recta entre sus puntos: cuanto más chica, más parecidos son en suelo y clima. Con 0, serían idénticos.

`nan_euclidean_distances` calcula esa distancia para todos los pares. Si a un condado le falta un dato, compara solo con las variables que tienen los dos y ajusta la escala, en vez de inventar el valor que falta.

Al final se calcula el **umbral de parecido**: la distancia por debajo de la cual está el `PERCENTIL_MAXIMO` % de todos los pares de la región.

In [6]:
if HAY_DATOS:
    distancias = pd.DataFrame(nan_euclidean_distances(x.to_numpy()), index=x.index, columns=x.index)

    pares = distancias.to_numpy()[np.triu_indices(len(distancias), k=1)]
    UMBRAL = np.percentile(pares, PERCENTIL_MAXIMO)
    print(f"Distancia entre condados: mediana {np.median(pares):.2f}, mínimo {pares.min():.2f}, máximo {pares.max():.2f}")
    print(f"Umbral de parecido (percentil {PERCENTIL_MAXIMO}): {UMBRAL:.2f}")
else:
    print("En espera de la tabla del punto 2.")

Distancia entre condados: mediana 1.28, mínimo 0.04, máximo 4.68
Umbral de parecido (percentil 10): 0.69


## 6. Los más parecidos de cada condado (kNN)

Para cada condado se ordenan todos los demás de menor a mayor distancia y se guardan los primeros `N_VECINOS`, **siempre que estén por debajo del umbral**. Eso es todo el kNN: no aprende nada, solo ordena por distancia.

Un condado no puede ser gemelo de sí mismo, así que se saca de su propia lista.

Al final se mira qué tan lejos quedan los parecidos **en el mapa** (km entre centroides) y cuántos son del mismo estado. Si casi todos están pegados, el parecido podría no estar diciendo mucho más que "es tu vecino": lo controla el paso 9.

In [7]:
if HAY_DATOS:
    filas = []
    for fips in distancias.index:
        cercanos = distancias.loc[fips].drop(fips).sort_values()
        cercanos = cercanos[cercanos <= UMBRAL].head(N_VECINOS)
        for orden, (vecino, dist) in enumerate(cercanos.items(), start=1):
            filas.append({"fips": fips, "orden": orden, "fips_vecino": vecino, "distancia": round(dist, 3)})
    vecinos = pd.DataFrame(filas)
    vecinos["mismo_estado"] = vecinos["fips"].str[:2] == vecinos["fips_vecino"].str[:2]
    if KM is not None:
        vecinos["km"] = [round(KM.at[f, v], 1) for f, v in zip(vecinos["fips"], vecinos["fips_vecino"])]

    cuantos = vecinos.groupby("fips").size().reindex(distancias.index, fill_value=0)
    print(f"Parecidos por condado: mediana {cuantos.median():.0f}, mínimo {cuantos.min()}, máximo {cuantos.max()}")
    print(f"Condados con menos de 3 parecidos (el front mostraría confianza baja): {(cuantos < 3).sum()}")
    print(f"De los parecidos, cuántos son del mismo estado: {vecinos['mismo_estado'].mean():.0%}")
    if KM is not None:
        print(f"Distancia en el mapa a los parecidos: mediana {vecinos['km'].median():.0f} km "
              f"(a los 5 primeros: {vecinos.loc[vecinos['orden'] <= 5, 'km'].median():.0f} km)")
else:
    print("En espera de la tabla del punto 2.")

Parecidos por condado: mediana 15, mínimo 0, máximo 15
Condados con menos de 3 parecidos (el front mostraría confianza baja): 11
De los parecidos, cuántos son del mismo estado: 68%
Distancia en el mapa a los parecidos: mediana 100 km (a los 5 primeros: 69 km)


## 7. En qué se parecen

El front muestra "Similar on soil drainage and July heat". Para cada par se toman las `N_SIMILAR_ON` variables en las que la diferencia (en la escala del paso 4) es más chica.

En los CSV se guardan como texto separado por `; `, en inglés para el front (`similar_on`) y en español para que el equipo revise (`similar_on_es`). En el JSON del front van como **lista** (`["soil drainage", "July heat"]`), que es lo que lee la pantalla.

In [8]:
if HAY_DATOS:
    def calcular_parecido(fips, vecino):
        diferencia = (z.loc[fips] - z.loc[vecino]).abs().dropna().sort_values()
        elegidas = diferencia.index[:N_SIMILAR_ON]
        return [ETIQUETAS[c][0] for c in elegidas], [ETIQUETAS[c][1] for c in elegidas]

    textos = [calcular_parecido(f, v) for f, v in zip(vecinos["fips"], vecinos["fips_vecino"])]
    vecinos["similar_on"] = ["; ".join(en) for en, _ in textos]
    vecinos["similar_on_es"] = ["; ".join(es) for _, es in textos]
    display(vecinos.head(8))
else:
    print("En espera de la tabla del punto 2.")

,fips,orden,fips_vecino,distancia,mismo_estado,km,similar_on,similar_on_es
0,17001,1,17067,0.223,True,46.3,soil drainage; May rain,drenaje del suelo; lluvia de mayo
1,17001,2,17185,0.250,True,335.2,soil drainage; soil organic carbon,drenaje del suelo; carbono del suelo
2,17001,3,19111,0.274,False,76.8,soil drainage; growing-season rain,drenaje del suelo; lluvia de la temporada
3,17001,4,17047,0.279,True,321.7,soil drainage; water the soil can hold,drenaje del suelo; agua que retiene el suelo
4,17001,5,17119,0.283,True,169.5,soil drainage; July heat,drenaje del suelo; calor de julio
5,17001,6,17117,0.296,True,135.1,soil drainage; growing-season heat,drenaje del suelo; calor de la temporada
6,17001,7,17005,0.298,True,194.0,soil drainage; July heat,drenaje del suelo; calor de julio
7,17001,8,29111,0.304,False,47.0,soil drainage; July heat,drenaje del suelo; calor de julio


## 8. Mirar ejemplos

Para los condados de `CONDADOS_EJEMPLO`: sus 5 parecidos y, al lado, sus valores de suelo y clima. Si el método anda, los números de los parecidos tienen que estar cerca de los del condado.

In [9]:
if HAY_DATOS:
    for fips in [f for f in CONDADOS_EJEMPLO if f in perfil.index]:
        suyos = vecinos[vecinos["fips"] == fips].head(5)
        comparar = perfil.loc[[fips] + suyos["fips_vecino"].tolist(), COLUMNAS].copy()
        comparar.insert(0, "condado", [nombrar(f) for f in comparar.index])
        comparar.insert(1, "distancia", [0.0] + suyos["distancia"].tolist())
        comparar.insert(2, "similar_on_es", [""] + suyos["similar_on_es"].tolist())
        print(f"\n{nombrar(fips)} y sus 5 parecidos:")
        display(comparar.round(2))
else:
    print("En espera de la tabla del punto 2.")


Story (Iowa) y sus 5 parecidos:


,condado,distancia,similar_on_es,aws0_100_mm,nccpi_maiz,soc0_30_gm2,pendiente_media_pct,clase_drenaje,prec_temporada_imerg_mm,gdd_temporada,tmax_jul_c,prec_imerg_mm_05
fips,,,,,,,,,,,,
19169,Story (Iowa),0.00,,182.6,0.73,9419.0,2.44,2.0,677.08,1738.46,28.86,110.86
19015,Boone (Iowa),0.19,calor de la temporada; calor de julio,179.1,0.69,9384.0,3.12,2.0,641.33,1738.46,28.86,108.59
18159,Tipton (Indiana),0.24,drenaje del suelo; lluvia de mayo,177.2,0.77,8630.0,1.30,2.0,684.30,1785.51,28.50,110.48
18007,Benton (Indiana),0.30,drenaje del suelo; carbono del suelo,171.3,0.77,9379.0,1.17,2.0,654.53,1830.86,28.42,113.13
17103,Lee (Illinois),0.31,drenaje del suelo; calor de la temporada,172.3,0.77,8262.0,2.08,2.0,691.34,1718.50,27.86,119.51
19069,Franklin (Iowa),0.34,drenaje del suelo; carbono del suelo,179.5,0.77,9617.0,2.11,2.0,644.98,1619.54,27.46,104.24



McLean (Illinois) y sus 5 parecidos:


,condado,distancia,similar_on_es,aws0_100_mm,nccpi_maiz,soc0_30_gm2,pendiente_media_pct,clase_drenaje,prec_temporada_imerg_mm,gdd_temporada,tmax_jul_c,prec_imerg_mm_05
fips,,,,,,,,,,,,
17113,McLean (Illinois),0.00,,172.4,0.78,8195.0,1.59,2.0,640.47,1826.20,28.67,134.20
17039,De Witt (Illinois),0.17,drenaje del suelo; pendiente,176.3,0.80,7329.0,1.57,2.0,647.25,1892.95,28.61,130.97
17115,Macon (Illinois),0.19,drenaje del suelo; pendiente,174.2,0.81,8850.0,1.62,2.0,667.10,1894.78,29.00,130.31
17045,Edgar (Illinois),0.24,drenaje del suelo; pendiente,171.8,0.80,7986.0,1.55,2.0,674.11,1941.16,28.85,126.21
17147,Piatt (Illinois),0.25,drenaje del suelo; agua que retiene el suelo,174.1,0.82,9372.0,1.22,2.0,665.24,1894.78,29.00,127.28
17107,Logan (Illinois),0.29,drenaje del suelo; pendiente,187.0,0.84,8085.0,1.64,2.0,655.62,1922.46,29.24,127.07



Sumner (Kansas) y sus 5 parecidos:


,condado,distancia,similar_on_es,aws0_100_mm,nccpi_maiz,soc0_30_gm2,pendiente_media_pct,clase_drenaje,prec_temporada_imerg_mm,gdd_temporada,tmax_jul_c,prec_imerg_mm_05
fips,,,,,,,,,,,,
20191,Sumner (Kansas),0.00,,166.6,0.46,4238.0,1.83,5.0,633.61,2312.59,33.61,130.52
20173,Sedgwick (Kansas),0.20,drenaje del suelo; pendiente,158.4,0.48,4626.0,1.61,5.0,623.50,2247.80,33.00,135.34
20077,Harper (Kansas),0.21,drenaje del suelo; calor de la temporada,154.1,0.45,4088.0,1.91,5.0,630.42,2315.04,34.21,123.54
20079,Harvey (Kansas),0.33,drenaje del suelo; pendiente,155.0,0.44,5173.0,1.61,5.0,588.71,2191.86,32.97,133.62
20029,Cloud (Kansas),0.42,drenaje del suelo; lluvia de mayo,174.3,0.51,5132.0,2.81,5.0,598.63,2086.20,33.07,125.66
20095,Kingman (Kansas),0.43,drenaje del suelo; calor de julio,139.7,0.44,3577.0,2.46,5.0,595.39,2220.58,33.58,119.28


## 9. ¿El parecido sirve? Primer control con el rinde

El rinde **no se usó** para buscar los parecidos. Por eso sirve como control: si dos condados se parecen de verdad en suelo y clima, el maíz les tiene que rendir parecido.

Para cada condado se toma su rinde típico de maíz (mediana 2009–2023) y se lo compara con el promedio del de 5 condados elegidos de distintas maneras. La diferencia es el **error**, en bu/ac: más chico, mejor. Se comparan cuatro maneras de elegir los 5:

- **Parecidos:** los 5 primeros de este punto.
- **Los 5 más cercanos en el mapa:** sin mirar suelo ni clima, solo la distancia entre centroides. **Es la vara que importa.** En agricultura lo que está cerca se parece, así que "mirá a tus vecinos" ya es un buen gemelo, gratis. Si los parecidos no le ganan, la búsqueda no aporta nada que no diga el mapa.
- **Al azar, mismo estado:** la regla "compará con tu estado". Ganarle es fácil.
- **Al azar, toda la región.**

El azar se repite 200 veces con semilla fija para que el resultado no dependa de la suerte.

**Cómo leerlo.** Acá compiten todos los condados, y lo esperable es que los parecidos y los vecinos del mapa queden cerca, porque muchas veces son los mismos condados. La prueba que importa para el front es la del **paso 19**: cuando el condado busca a los que **sí hacen algo que él no hace**, los vecinos del mapa que lo hacen pueden estar lejos y ser distintos. Ahí es donde los gemelos tienen que ganar.

In [10]:
if HAY_DATOS:
    rinde = tabla[tabla["cultivo"] == "maiz"].groupby("fips")["rend_bu_ac"].median()
    azar = np.random.default_rng(0)

    def error_contra(grupos):
        """Diferencia media (bu/ac) entre el rinde del condado y el promedio del de su grupo."""
        return np.nanmean([abs(rinde.get(f, np.nan) - rinde.reindex(g).mean()) for f, g in grupos.items()])

    def cercanos_en_mapa(fips, candidatos, n):
        """Los n condados de `candidatos` más cercanos a `fips` en el mapa (sin contarlo a él)."""
        return KM.loc[fips, [c for c in candidatos if c != fips]].nsmallest(n).index.tolist()

    top5 = vecinos[vecinos["orden"] <= 5].groupby("fips")["fips_vecino"].apply(list)
    top5 = top5[top5.str.len() >= 3]
    por_estado = perfil.groupby("estado").groups

    def al_azar(mismo_estado):
        grupos = {}
        for f in top5.index:
            candidatos = [c for c in (por_estado[f[:2]] if mismo_estado else perfil.index) if c != f]
            grupos[f] = list(azar.choice(candidatos, size=5, replace=False))
        return error_contra(grupos)

    errores = {"Parecidos (este punto)": error_contra(top5)}
    if KM is not None:
        errores["Los 5 más cercanos en el mapa"] = error_contra({f: cercanos_en_mapa(f, perfil.index, 5) for f in top5.index})
    else:
        print("Sin la entrega no hay centroides: falta el control con los vecinos del mapa.")
    errores["Al azar, mismo estado"] = np.mean([al_azar(True) for _ in range(200)])
    errores["Al azar, toda la región"] = np.mean([al_azar(False) for _ in range(200)])
    control_todos = pd.Series(errores, name="diferencia media de rinde de maíz (bu/ac)")

    print(f"Condados evaluados: {len(top5)} (los que tienen al menos 3 parecidos) | lluvia: {LLUVIA.upper()}")
    display(control_todos.round(1).to_frame())
    if KM is not None:
        diferencia = control_todos["Parecidos (este punto)"] - control_todos["Los 5 más cercanos en el mapa"]
        veredicto = ("empatan: el parecido elige casi lo mismo que el mapa" if abs(diferencia) < 0.5
                     else "ganan los parecidos" if diferencia < 0 else "los parecidos no le ganan al mapa")
        print(f"Parecidos − vecinos del mapa: {diferencia:+.2f} bu/ac ({veredicto})")
else:
    print("En espera de la tabla del punto 2.")

Condados evaluados: 884 (los que tienen al menos 3 parecidos) | lluvia: IMERG


,diferencia media de rinde de maíz (bu/ac)
Parecidos (este punto),10.6
Los 5 más cercanos en el mapa,8.8
"Al azar, mismo estado",18.4
"Al azar, toda la región",26.3


Parecidos − vecinos del mapa: +1.76 bu/ac (los parecidos no le ganan al mapa)


## 10. Guardar

`backend/Datos/gemelos_vecinos.csv`: una fila por condado y parecido. Es la entrada de la parte 2.

| Columna | Qué es |
|---|---|
| `fips` | El condado |
| `orden` | 1 = el más parecido |
| `fips_vecino` | El condado parecido |
| `distancia` | Qué tan distintos son en suelo y clima (0 = idénticos) |
| `mismo_estado` | Si están en el mismo estado |
| `km` | Distancia en el mapa entre los centroides |
| `similar_on`, `similar_on_es` | Las variables en las que más se parecen, separadas por `; ` |

In [11]:
if HAY_DATOS:
    ruta = SALIDA / "gemelos_vecinos.csv"
    vecinos.to_csv(ruta, index=False)
    print(f"Guardado {ruta.relative_to(RAIZ)}: {len(vecinos)} filas, {vecinos['fips'].nunique()} condados")
else:
    print("En espera de la tabla del punto 2.")

Guardado backend/Datos/gemelos_vecinos.csv: 13028 filas, 893 condados


# Parte 2 · Los gemelos de cada opción

La pregunta del productor: *"si hago algo que en mi condado casi no se hace, ¿cómo les fue a los condados como el mío que sí lo hacen?"*

Hay dos casos:

1. **Una rotación que el condado casi no hace** (pasos 11 a 17): todos los condados siembran maíz y soja, pero en algunos casi nadie repite maíz sobre maíz, y en otros casi nadie lo rota. Las opciones son las mismas del punto 2.
2. **Un cultivo que el condado no siembra** (trigo de invierno, avena): paso 18.

| Opción (punto 2) | Id en el front | Qué es | Qué gemelo cuenta como "lo hizo" |
|---|---|---|---|
| maíz `actual` | `corn` | Seguir como el condado | Cualquier parecido |
| maíz `rotado` | `corn_after_soy` | Casi nada de maíz sobre maíz | Está en el 25 % de condados de la región que **menos** repite maíz |
| maíz `continuo` | `corn_after_corn` | Mucho maíz sobre maíz | Está en el 25 % que **más** lo repite |
| soja `actual`, `rotado`, `continuo` | `soy`, `soy_after_corn`, `soy_after_soy` | Igual, con soja | Igual, con soja |
| trigo `nuevo` | `wheat` | Sembrar trigo de invierno | NASS publica su rinde en al menos 5 años |
| avena `nuevo` | `oats` | Sembrar avena | Igual |

El paso 19 controla que los gemelos le ganen a la alternativa simple (los más cercanos en el mapa que hacen la opción), y el 20 guarda todo.

**Ojo, lo más importante de esta parte:** el rinde de NASS es **del condado entero**, no del lote. Si un gemelo hace mucho maíz sobre maíz, su rinde mezcla esos lotes con los rotados. El rango de los gemelos dice "a los condados como el tuyo que hacen esto les fue así", **no** "si hacés esto vas a rendir así". El paso 16 chequea si las dos cosas coinciden.

## 11. Parámetros de la parte 2

- `ANIO_TENDENCIA`: el rinde sube año a año por la tecnología. Para comparar 2009 con 2023, cada rinde se lleva al nivel de tecnología de 2025, el mismo año que usa el punto 2. Así el rango de los gemelos se puede poner al lado del rango del modelo.
- `CUANTIL_OPCION`: cuánto tiene que hacer un gemelo la opción para contar. 0,25 = estar en el 25 % de la región que más (o menos) lo hace.
- `N_MOSTRAR` y `MIN_GEMELOS`: el front muestra hasta 5 gemelos; con menos de 3, la confianza es baja (regla de `docs/front-end-definido.md`) y **no se da rango**.
- `MIN_ANIOS_ALTA`: para confianza alta, además de 5 gemelos, hace falta un mínimo de años con rinde entre todos.
- `OPCION_FRONT`, `CULTIVO_FRONT` y `ANTECESOR_FRONT`: cómo se cruzan las opciones de ML con el front. Los cultivos usan los ids de `web/src/ui/crops.ts` (`corn`, `soy`, `wheat`…). La avena no tiene id propio ahí, así que va como `other` ("Other crops").

In [12]:
ANIO_TENDENCIA = 2025
CUANTIL_OPCION = 0.25
N_MOSTRAR = 5
MIN_GEMELOS = 3
MIN_ANIOS_ALTA = 40

COLUMNA_MISMO = {"maiz": "pct_maiz_tras_maiz", "soja": "pct_soja_tras_soja"}
OPCIONES = ["actual", "rotado", "continuo"]
CULTIVO_EN = {"maiz": "corn", "soja": "soybean"}
# Cómo se nombra, en la frase del front, a los gemelos que hicieron cada opción
OPCION_EN = {
    ("maiz", "actual"): "Counties like yours",
    ("maiz", "rotado"): "Counties like yours that rarely grow corn after corn",
    ("maiz", "continuo"): "Counties like yours that grow a lot of corn after corn",
    ("soja", "actual"): "Counties like yours",
    ("soja", "rotado"): "Counties like yours that rarely grow soybean after soybean",
    ("soja", "continuo"): "Counties like yours that grow a lot of soybean after soybean",
}
RUTA_MODELO_P2 = SALIDA / "escenarios_rotacion_resumen.csv"

# Cruce con el front (web/src/next/types.ts y web/src/ui/crops.ts)
CULTIVO_FRONT = {"maiz": "corn", "soja": "soy", "trigo_invierno": "wheat", "avena": "other"}
OPCION_FRONT = {
    ("maiz", "actual"): "corn", ("maiz", "rotado"): "corn_after_soy", ("maiz", "continuo"): "corn_after_corn",
    ("soja", "actual"): "soy", ("soja", "rotado"): "soy_after_corn", ("soja", "continuo"): "soy_after_soy",
    ("trigo_invierno", "nuevo"): "wheat", ("avena", "nuevo"): "oats",
}
# El cultivo anterior de cada opción (None = no se fija)
ANTECESOR_FRONT = {("maiz", "rotado"): "soy", ("maiz", "continuo"): "corn",
                   ("soja", "rotado"): "corn", ("soja", "continuo"): "soy"}

## 12. Llevar cada rinde a la tecnología de hoy

Para cada estado y cultivo se ajusta una recta rinde ~ año con 2009–2023 (la **tendencia**). La pendiente dice cuánto sube el rinde por año por la tecnología. Después, a cada rinde se le suma lo que le faltaba para llegar a 2025:

> rinde ajustado = rinde del año + pendiente × (2025 − año)

Ejemplo: si en Iowa el maíz sube 2 bu/ac por año, un rinde de 170 en 2012 pasa a 170 + 2 × 13 = 196. La sequía de 2012 se sigue viendo (queda bajo), pero ya no se confunde con "eran otros tiempos".

No se usa nada de 2024–2025: son la prueba del punto 2.

In [13]:
if HAY_DATOS:
    rindes = tabla[tabla["rend_bu_ac"].notna()][["fips", "anio", "estado", "cultivo", "rend_bu_ac"]].copy()
    pendientes = {}
    for (cultivo, estado), grupo in rindes.groupby(["cultivo", "estado"]):
        pendiente, _ = np.polyfit(grupo["anio"], grupo["rend_bu_ac"], 1)
        pendientes[(cultivo, estado)] = pendiente
        rindes.loc[grupo.index, "rend_hoy_bu_ac"] = grupo["rend_bu_ac"] + pendiente * (ANIO_TENDENCIA - grupo["anio"])

    display(pd.Series(pendientes, name="bu/ac por año").rename_axis(["cultivo", "estado"]).round(2).to_frame())
else:
    print("En espera de la tabla del punto 2.")

bu/ac por año
cultivo estado               
maiz    17               4.28
        18               3.75
        19               3.26
        20               0.68
        27               1.70
        29               2.73
        31               1.67
        39               2.47
        46               1.24
        55               2.31
soja    17               1.28
        18               1.02
        19               0.82
        20               0.12
        27               0.53
        29               0.77
        31               0.32
        39               0.61
        46               0.36
        55               0.70

## 13. Qué hace cada condado

Para cada condado y cultivo, el % típico del cultivo sembrado sobre sí mismo (la mediana de 2009–2023). Con eso se calculan los dos cortes:

- **rotado:** el condado está por debajo del cuantil 25 % de la región;
- **continuo:** está por encima del cuantil 75 %.

También se marca si **el propio condado** ya está en esa opción: en ese caso la opción no es nueva para él.

In [14]:
if HAY_DATOS:
    practica = {}
    cortes = {}
    for cultivo, columna in COLUMNA_MISMO.items():
        pct = tabla[tabla["cultivo"] == cultivo].groupby("fips")[columna].median()
        practica[cultivo] = pct
        cortes[cultivo] = {"rotado": pct.quantile(CUANTIL_OPCION), "continuo": pct.quantile(1 - CUANTIL_OPCION)}
        print(f"{cultivo}: % sobre sí mismo, mediana {pct.median():.1f} %. "
              f"Rotado ≤ {cortes[cultivo]['rotado']:.1f} % | continuo ≥ {cortes[cultivo]['continuo']:.1f} %")

    def hace_la_opcion(cultivo, opcion, fips):
        pct = practica[cultivo].get(fips, np.nan)
        if opcion == "actual":
            return True
        if np.isnan(pct):
            return False
        return pct <= cortes[cultivo]["rotado"] if opcion == "rotado" else pct >= cortes[cultivo]["continuo"]
else:
    print("En espera de la tabla del punto 2.")

maiz: % sobre sí mismo, mediana 22.0 %. Rotado ≤ 12.1 % | continuo ≥ 38.7 %
soja: % sobre sí mismo, mediana 13.3 %. Rotado ≤ 5.9 % | continuo ≥ 27.5 %


## 14. Los gemelos de cada opción

Para cada condado, cultivo y opción se recorren **todos** los condados por debajo del umbral de parecido (paso 5), del más parecido al menos, y se quedan los primeros `N_MOSTRAR` que hicieron la opción.

Se usan todos los que pasan el umbral y no solo los 15 de la parte 1: los condados que hacen mucho maíz sobre maíz suelen estar juntos en una zona, y con solo 15 vecinos muchos condados se quedaban sin ningún gemelo.

De cada gemelo se guarda:

- `similar_on`: en qué se parece;
- `pct_sobre_si_mismo`: cuánto hace la opción (% del cultivo sembrado sobre sí mismo);
- `pct_area`: qué parte del área sembrada ocupa ese cultivo (paso 2). Es lo que el front muestra como "Corn on 48% of fields". Antes las dos cosas iban en una sola columna y el front habría leído el % de maíz sobre maíz como "% de los lotes con maíz";
- `rend_tipico_bu_ac`: su rinde típico (mediana del rinde llevado a hoy).

In [15]:
if HAY_DATOS:
    rinde_tipico = rindes.groupby(["cultivo", "fips"])["rend_hoy_bu_ac"].median()

    def parte_del_area(cultivo, fips):
        """% del área sembrada que ocupa el cultivo en el condado (NaN si no hay entrega o dato)."""
        return round(PARTE_AREA[cultivo].get(fips, np.nan), 1) if cultivo in PARTE_AREA else np.nan

    filas = []
    for fips in distancias.index:
        cercanos = distancias.loc[fips].drop(fips).sort_values()
        cercanos = cercanos[cercanos <= UMBRAL]
        for cultivo in COLUMNA_MISMO:
            for opcion in OPCIONES:
                elegidos = [v for v in cercanos.index if hace_la_opcion(cultivo, opcion, v)][:N_MOSTRAR]
                for orden, vecino in enumerate(elegidos, start=1):
                    en, es = calcular_parecido(fips, vecino)
                    filas.append({
                        "fips": fips, "cultivo": cultivo, "rotacion": opcion, "orden": orden,
                        "fips_gemelo": vecino, "distancia": round(cercanos[vecino], 3),
                        "similar_on": "; ".join(en), "similar_on_es": "; ".join(es),
                        "pct_sobre_si_mismo": round(practica[cultivo].get(vecino, np.nan), 1),
                        "pct_area": parte_del_area(cultivo, vecino),
                        "rend_tipico_bu_ac": round(rinde_tipico.get((cultivo, vecino), np.nan), 1),
                    })
    gemelos = pd.DataFrame(filas)

    cuantos = (gemelos.groupby(["fips", "cultivo", "rotacion"]).size()
               .reindex(pd.MultiIndex.from_product([distancias.index, list(COLUMNA_MISMO), OPCIONES],
                                                   names=["fips", "cultivo", "rotacion"]), fill_value=0))
    cobertura = cuantos.groupby(["cultivo", "rotacion"]).agg(
        sin_gemelos=lambda s: (s == 0).mean(), menos_de_3=lambda s: (s < MIN_GEMELOS).mean(),
        con_5=lambda s: (s >= N_MOSTRAR).mean())
    print("Proporción de condados según cuántos gemelos tienen:")
    display((cobertura * 100).round(0).astype(int).astype(str) + " %")
else:
    print("En espera de la tabla del punto 2.")

Proporción de condados según cuántos gemelos tienen:


sin_gemelos menos_de_3 con_5
cultivo rotacion                             
maiz    actual           0 %        1 %  98 %
        continuo         5 %       15 %  75 %
        rotado          12 %       21 %  74 %
soja    actual           0 %        1 %  98 %
        continuo        27 %       37 %  60 %
        rotado          20 %       26 %  69 %

## 15. El rango: p10, mediana y p90

Para cada condado, cultivo y opción se juntan **todos los años de todos sus gemelos** (rinde llevado a hoy). Con 5 gemelos y 15 años son unos 75 valores. De ahí salen:

- **p10, año malo:** solo 1 de cada 10 valores quedó por debajo.
- **mediana, año típico:** la mitad quedó arriba y la mitad abajo.
- **p90, año bueno:** solo 1 de cada 10 quedó por encima.

Es la misma lectura que el rango del punto 2, pero en vez de salir de un modelo sale de lo que **pasó de verdad** en condados parecidos.

**Solo se da rango con al menos `MIN_GEMELOS` gemelos.** Con uno o dos, el "rango" sería el de esos pocos condados y parecería más firme de lo que es. Los gemelos se muestran igual, pero el rango queda vacío ("Missing data").

**Confianza**, con su motivo en inglés para el front. El front solo conoce `high`, `medium` y `low`:

- **high:** 5 gemelos y al menos `MIN_ANIOS_ALTA` años con rinde;
- **medium:** 3 o 4 gemelos;
- **low:** 1 o 2 gemelos, sin rango;
- **low, sin gemelos:** ningún gemelo hizo esa opción. El motivo empieza con "Missing data" y no hay rango.

Con `armar_rango` se arma igual la fila de cualquier opción, también las de trigo y avena del paso 18.

In [16]:
if HAY_DATOS:
    rinde_por_condado = {k: g["rend_hoy_bu_ac"].to_numpy() for k, g in rindes.groupby(["cultivo", "fips"])}

    def medir_confianza(n_gemelos, n_anios, cultivo, opcion):
        quienes = OPCION_EN[(cultivo, opcion)].replace("Counties like yours", "counties like yours")
        if n_gemelos == 0:
            return "low", "Missing data: no county with similar soil and weather does this."
        if n_gemelos < MIN_GEMELOS:
            if n_gemelos == 1:
                quienes = quienes.replace("counties", "county").replace(" grow ", " grows ")
            return "low", f"Low confidence: only {n_gemelos} {quienes}, too few for a yield range."
        if n_gemelos >= N_MOSTRAR and n_anios >= MIN_ANIOS_ALTA:
            return "high", f"{n_gemelos} {quienes}, {n_anios} years of yield data."
        return "medium", f"{n_gemelos} {quienes}, {n_anios} years of yield data."

    def armar_rango(fips, cultivo, opcion, elegidos, ya_lo_hace):
        """Una fila de gemelos_rango. El rango junta todos los años de los gemelos, y solo se da con MIN_GEMELOS o más."""
        valores = np.concatenate([rinde_por_condado.get((cultivo, g), []) for g in elegidos] or [[]])
        nivel, motivo = medir_confianza(len(elegidos), len(valores), cultivo, opcion)
        hay_rango = len(elegidos) >= MIN_GEMELOS and len(valores) > 0
        return {
            "fips": fips, "cultivo": cultivo, "rotacion": opcion, "n_gemelos": len(elegidos), "n_anios": len(valores),
            "p10": np.percentile(valores, 10) if hay_rango else np.nan,
            "mediana": np.median(valores) if hay_rango else np.nan,
            "p90": np.percentile(valores, 90) if hay_rango else np.nan,
            "confianza": nivel, "motivo": motivo, "condado_ya_lo_hace": bool(ya_lo_hace), "usar_como_rango": hay_rango,
        }

    filas = []
    for (fips, cultivo, opcion), n in cuantos.items():
        suyos = gemelos[(gemelos["fips"] == fips) & (gemelos["cultivo"] == cultivo) & (gemelos["rotacion"] == opcion)]
        filas.append(armar_rango(fips, cultivo, opcion, suyos["fips_gemelo"].tolist(), hace_la_opcion(cultivo, opcion, fips)))
    rango = pd.DataFrame(filas).round({"p10": 1, "mediana": 1, "p90": 1})
    display(rango.groupby(["cultivo", "rotacion"])["confianza"].value_counts(normalize=True)
            .mul(100).round(0).unstack(fill_value=0).astype(int).astype(str) + " %")
else:
    print("En espera de la tabla del punto 2.")

confianza         high   low medium
cultivo rotacion                   
maiz    actual    96 %   1 %    3 %
        continuo  74 %  15 %   11 %
        rotado    72 %  21 %    6 %
soja    actual    89 %   1 %   10 %
        continuo  60 %  37 %    3 %
        rotado    58 %  26 %   16 %

## 16. Chequeo: ¿los gemelos cuentan lo mismo que el modelo del punto 2?

El punto 2 midió el efecto de rotar comparando a cada condado **consigo mismo** (efectos fijos), y da que el cultivo continuo rinde menos, como dicen los experimentos de campo. Los gemelos comparan **condados distintos**. Si las dos cosas apuntan para el mismo lado, el rango de los gemelos se puede mostrar como evidencia de la opción.

Se compara la mediana de `continuo` − la de `rotado`, con los gemelos y con el modelo, en los condados que tienen las dos opciones con al menos 3 gemelos.

Si con los gemelos `continuo` rinde **más** que `rotado`, al revés que el modelo y los experimentos, no se puede mostrar como "lo que te pasaría si hacés esto". Lo más probable es que los condados que hacen mucho maíz sobre maíz sean los más productivos por otras razones (el punto 2 encontró lo mismo antes de usar efectos fijos). En ese caso, en las opciones `rotado` y `continuo` de ese cultivo:

- `usar_como_rango = False`: los gemelos se muestran como "quiénes lo hacen", pero el rango de la opción sale del punto 2;
- la confianza de los gemelos baja a `low`, con el motivo;
- el texto del front **no** muestra el rango de los gemelos (paso 20). Antes lo mostraba igual, y con confianza alta.

In [17]:
if HAY_DATOS:
    if RUTA_MODELO_P2.exists():
        modelo_p2 = pd.read_csv(RUTA_MODELO_P2, dtype={"fips": str})
        juntos = rango.merge(modelo_p2, on=["fips", "cultivo", "rotacion"], how="left")
        con_dato = juntos[juntos["n_gemelos"] >= MIN_GEMELOS]

        ancho = con_dato.pivot_table(index=["cultivo", "fips"], columns="rotacion", values=["mediana", "mediana_clima"])
        ancho = ancho.dropna(subset=[("mediana", "continuo"), ("mediana", "rotado")])
        diferencia = pd.DataFrame({
            "gemelos: continuo − rotado": ancho[("mediana", "continuo")] - ancho[("mediana", "rotado")],
            "modelo p2: continuo − rotado": ancho[("mediana_clima", "continuo")] - ancho[("mediana_clima", "rotado")],
        })
        resumen = diferencia.groupby("cultivo").median().round(1)
        resumen["condados"] = diferencia.groupby("cultivo").size()
        display(resumen)

        for cultivo, fila in resumen.iterrows():
            mismo_lado = np.sign(fila["gemelos: continuo − rotado"]) == np.sign(fila["modelo p2: continuo − rotado"])
            print(f"{cultivo}: {'OK, apuntan para el mismo lado' if mismo_lado else 'ATENCIÓN: los gemelos apuntan al revés que el modelo'}")
            if not mismo_lado:
                cambiar = (rango["cultivo"] == cultivo) & (rango["rotacion"] != "actual") & (rango["n_gemelos"] > 0)
                rango.loc[cambiar, "usar_como_rango"] = False
                rango.loc[cambiar, "confianza"] = "low"
                # El punto 2 no compara rotaciones fuera de la zona maíz-soja: ahí no hay rango del modelo
                tiene_modelo = (rango.merge(modelo_p2[["fips", "cultivo", "rotacion"]].drop_duplicates(), how="left", indicator=True)
                                ["_merge"].eq("both").to_numpy())
                rango.loc[cambiar & tiene_modelo, "motivo"] = ("Low confidence: these counties differ from yours in more than "
                                                               "this rotation, so their yield doesn't show its effect. The range comes from the model.")
                rango.loc[cambiar & ~tiene_modelo, "motivo"] = ("Low confidence: these counties differ from yours in more than "
                                                                "this rotation, and the model has no range for this option here.")

        print("\nMediana del rango por opción, gemelos contra modelo del punto 2:")
        display(juntos.groupby(["cultivo", "rotacion"])[["p10", "mediana", "p90", "p10_clima", "mediana_clima", "p90_clima"]]
                .median().round(1))
    else:
        print("No está escenarios_rotacion_resumen.csv del punto 2: el chequeo queda en espera.")
else:
    print("En espera de la tabla del punto 2.")

,gemelos: continuo − rotado,modelo p2: continuo − rotado,condados
cultivo,,,
maiz,17.9,-33.6,583
soja,-8.9,-5.0,406


maiz: ATENCIÓN: los gemelos apuntan al revés que el modelo
soja: OK, apuntan para el mismo lado

Mediana del rango por opción, gemelos contra modelo del punto 2:


p10  mediana    p90  p10_clima  mediana_clima  p90_clima
cultivo rotacion                                                            
maiz    actual    156.8    190.2  213.8      179.1          189.8      197.4
        continuo  177.6    206.0  229.8      163.0          174.2      182.9
        rotado    145.4    187.4  210.7      196.6          207.9      216.4
soja    actual     47.3     56.3   63.6       52.3           56.0       58.0
        continuo   47.2     54.7   60.4       51.4           55.2       57.0
        rotado     53.9     62.6   68.1       56.5           60.3       62.1

## 17. Ejemplo

Lo que vería un productor de `CONDADOS_EJEMPLO[0]` (Story, Iowa) en maíz: el rango de cada opción según los gemelos, al lado del rango del modelo (`*_clima`), y los gemelos de la opción que **su condado no hace**.

In [18]:
if HAY_DATOS:
    fips = CONDADOS_EJEMPLO[0]
    ver = rango[(rango["fips"] == fips) & (rango["cultivo"] == "maiz")]
    if RUTA_MODELO_P2.exists():
        ver = ver.merge(modelo_p2, on=["fips", "cultivo", "rotacion"], how="left")
    display(ver.drop(columns=["fips", "cultivo"]))

    nuevas = ver.loc[~ver["condado_ya_lo_hace"], "rotacion"].tolist()
    if nuevas:
        print(f"Gemelos de la opción '{nuevas[0]}', que {nombrar(fips)} no hace:")
        ejemplo = gemelos[(gemelos["fips"] == fips) & (gemelos["cultivo"] == "maiz") & (gemelos["rotacion"] == nuevas[0])]
        display(ejemplo.assign(gemelo=ejemplo["fips_gemelo"].map(nombrar))
                .drop(columns=["fips", "cultivo", "rotacion", "similar_on"]))
else:
    print("En espera de la tabla del punto 2.")

,rotacion,n_gemelos,n_anios,p10,mediana,p90,confianza,motivo,condado_ya_lo_hace,usar_como_rango,mediana_clima,p10_clima,p90_clima
0,actual,5,75,199.5,222.7,238.8,high,"5 counties like yours, 75 years of yield data.",True,True,214.0,205.2,221.0
1,rotado,5,74,187.6,215.7,241.9,low,Low confidence: these counties differ from you...,False,False,225.7,216.9,232.7
2,continuo,5,74,192.7,220.2,236.6,low,Low confidence: these counties differ from you...,False,False,192.1,183.3,199.1


Gemelos de la opción 'rotado', que Story (Iowa) no hace:


,orden,fips_gemelo,distancia,similar_on_es,pct_sobre_si_mismo,pct_area,rend_tipico_bu_ac,gemelo
7880,1,18159,0.238,drenaje del suelo; lluvia de mayo,11.2,48.4,232.3,Tipton (Indiana)
7881,2,18067,0.433,productividad del suelo; lluvia de la temporada,8.7,47.5,224.4,Howard (Indiana)
7882,3,18011,0.436,lluvia de la temporada; lluvia de mayo,10.0,49.2,214.4,Boone (Indiana)
7883,4,18095,0.477,lluvia de mayo; lluvia de la temporada,10.8,46.7,215.1,Madison (Indiana)
7884,5,18059,0.509,lluvia de mayo; calor de la temporada,10.6,46.2,206.2,Hancock (Indiana)


## 18. Cultivos que el condado no siembra: trigo de invierno y avena

Es el caso más claro de gemelo: *"en mi condado nadie siembra trigo; ¿cómo les fue a los condados como el mío que sí lo siembran?"*. Los rindes vienen de `tabla_anual.csv` de la entrega (NASS, por condado).

- **Quién "siembra" el cultivo:** NASS publica su rinde en al menos `MIN_ANIOS_CULTIVO` de los años 2009–2023. NASS solo publica un condado cuando el cultivo tiene superficie suficiente, así que es una buena señal de que se siembra de verdad. El propio condado se marca con `condado_ya_lo_hace` si lo cumple.
- **`pct_area`:** qué parte del área sembrada ocupa ese cultivo (paso 2), igual que en maíz y soja. `pct_sobre_si_mismo` queda vacío: no aplica.
- **Tendencia:** la misma cuenta del paso 12, por estado. Si un estado tiene menos de `MIN_FILAS_TENDENCIA` filas de ese cultivo, se usa la recta de toda la región.
- La opción se llama `nuevo` en los CSV; en el front, `wheat` y `oats`.
- **No hay modelo del punto 2 para estos cultivos**, así que no hay chequeo como el del paso 16. El rango sale solo de los gemelos y vale la misma advertencia: es el rinde de condados que lo siembran, no de tu lote.

In [19]:
CULTIVOS_NUEVOS = {"trigo_invierno": "winter wheat", "avena": "oats"}
MIN_ANIOS_CULTIVO = 5
MIN_FILAS_TENDENCIA = 30
SIEMBRAN = {}   # cultivo -> condados que lo siembran (lo usa el control del paso 19)

if HAY_DATOS and HAY_ENTREGA:
    filas_g, filas_r = [], []
    for cultivo, nombre_en in CULTIVOS_NUEVOS.items():
        CULTIVO_EN[cultivo] = nombre_en
        OPCION_EN[(cultivo, "nuevo")] = f"Counties like yours that grow {nombre_en}"

        d = anual[(anual["cultivo"] == cultivo) & anual["rend_bu_ac"].notna()].copy()
        recta_region = np.polyfit(d["anio"], d["rend_bu_ac"], 1)[0]
        for estado, grupo in d.groupby("estado"):
            pendiente = np.polyfit(grupo["anio"], grupo["rend_bu_ac"], 1)[0] if len(grupo) >= MIN_FILAS_TENDENCIA else recta_region
            d.loc[grupo.index, "rend_hoy_bu_ac"] = grupo["rend_bu_ac"] + pendiente * (ANIO_TENDENCIA - grupo["anio"])

        anios = d.groupby("fips").size()
        siembran = set(anios[anios >= MIN_ANIOS_CULTIVO].index)
        SIEMBRAN[cultivo] = siembran
        rinde_por_condado.update({(cultivo, f): g["rend_hoy_bu_ac"].to_numpy() for f, g in d.groupby("fips")})
        tipico = d.groupby("fips")["rend_hoy_bu_ac"].median()
        print(f"{nombre_en}: lo siembran {len(siembran)} condados "
              f"({pd.Series([f[:2] for f in siembran]).map(NOMBRES_ESTADO).value_counts().to_dict()})")

        for fips in distancias.index:
            cercanos = distancias.loc[fips].drop(fips).sort_values()
            cercanos = cercanos[cercanos <= UMBRAL]
            elegidos = [v for v in cercanos.index if v in siembran][:N_MOSTRAR]
            for orden, vecino in enumerate(elegidos, start=1):
                en, es = calcular_parecido(fips, vecino)
                filas_g.append({
                    "fips": fips, "cultivo": cultivo, "rotacion": "nuevo", "orden": orden, "fips_gemelo": vecino,
                    "distancia": round(cercanos[vecino], 3), "similar_on": "; ".join(en), "similar_on_es": "; ".join(es),
                    "pct_sobre_si_mismo": np.nan, "pct_area": parte_del_area(cultivo, vecino),
                    "rend_tipico_bu_ac": round(tipico.get(vecino, np.nan), 1),
                })
            filas_r.append(armar_rango(fips, cultivo, "nuevo", elegidos, fips in siembran))

    nuevos_g = pd.DataFrame(filas_g)
    nuevos_r = pd.DataFrame(filas_r).round({"p10": 1, "mediana": 1, "p90": 1})
    gemelos = pd.concat([gemelos[gemelos["rotacion"] != "nuevo"], nuevos_g], ignore_index=True)
    rango = pd.concat([rango[rango["rotacion"] != "nuevo"], nuevos_r], ignore_index=True)

    no_lo_siembran = nuevos_r[~nuevos_r["condado_ya_lo_hace"]]
    print("\nCondados que NO siembran el cultivo, según la confianza de sus gemelos:")
    display((no_lo_siembran.groupby("cultivo")["confianza"].value_counts(normalize=True) * 100)
            .round(0).unstack(fill_value=0).astype(int).astype(str) + " %")

    fips = CONDADOS_EJEMPLO[0]
    print(f"\nEjemplo: {nombrar(fips)}")
    display(nuevos_r[nuevos_r["fips"] == fips].drop(columns=["fips", "rotacion"]))
    ejemplo = nuevos_g[(nuevos_g["fips"] == fips) & (nuevos_g["cultivo"] == "trigo_invierno")]
    display(ejemplo.assign(gemelo=ejemplo["fips_gemelo"].map(nombrar))
            .drop(columns=["fips", "cultivo", "rotacion", "similar_on"]))
elif HAY_DATOS:
    print(f"En espera de la entrega {VERSION_ENTREGA}: sin trigo ni avena.")
else:
    print("En espera de la tabla del punto 2.")

winter wheat: lo siembran 549 condados ({'Kansas': 103, 'Illinois': 87, 'Indiana': 75, 'Missouri': 74, 'Ohio': 72, 'Nebraska': 56, 'Wisconsin': 44, 'South Dakota': 38})


oats: lo siembran 253 condados ({'Minnesota': 64, 'Wisconsin': 58, 'Iowa': 50, 'South Dakota': 41, 'Ohio': 14, 'Nebraska': 13, 'Illinois': 11, 'Kansas': 2})



Condados que NO siembran el cultivo, según la confianza de sus gemelos:


confianza,high,low,medium
cultivo,,,
avena,33 %,30 %,37 %
trigo_invierno,70 %,14 %,16 %



Ejemplo: Story (Iowa)


,cultivo,n_gemelos,n_anios,p10,mediana,p90,confianza,motivo,condado_ya_lo_hace,usar_como_rango
278,trigo_invierno,5,46,76.5,89.9,101.7,high,"5 counties like yours that grow winter wheat, ...",False,True
1173,avena,5,34,64.2,82.8,101.6,medium,"5 counties like yours that grow oats, 34 years...",True,True


,orden,fips_gemelo,distancia,similar_on_es,pct_sobre_si_mismo,pct_area,rend_tipico_bu_ac,gemelo
1380,1,18159,0.238,drenaje del suelo; lluvia de mayo,NaN,1.1,92.5,Tipton (Indiana)
1381,2,17103,0.309,drenaje del suelo; calor de la temporada,NaN,0.3,90.0,Lee (Illinois)
1382,3,17139,0.414,drenaje del suelo; lluvia de la temporada,NaN,0.6,82.9,Moultrie (Illinois)
1383,4,17019,0.415,drenaje del suelo; calor de julio,NaN,0.5,89.8,Champaign (Illinois)
1384,5,17041,0.421,drenaje del suelo; calor de julio,NaN,0.4,92.8,Douglas (Illinois)


## 19. Segundo control: ¿los gemelos le ganan a los vecinos del mapa que hacen la opción?

Es la prueba que importa para el front. Un condado que casi no repite maíz quiere ver a los condados **como el suyo** que sí lo repiten. La alternativa simple es "los condados más cercanos en el mapa que lo repiten". Si los gemelos no le ganan a eso, la promesa de "suelo y clima parecidos, aunque estén lejos" no se cumple.

Para cada condado que **no** hace la opción y tiene al menos `MIN_GEMELOS` gemelos de esa opción:

- **Gemelos:** sus gemelos de esa opción (pasos 14 y 18).
- **Mapa:** la misma cantidad de condados que hacen la opción, los más cercanos en el mapa.

Se mide igual que en el paso 9: la diferencia entre el rinde típico de maíz del condado y el promedio del de cada grupo. Se usa el maíz en todas las opciones porque es el único cultivo que tienen casi todos los condados: acá mide **qué tan parecido es el lugar**, no cómo rinde la opción. También se muestra a cuántos km quedan unos y otros.

Si en alguna opción los gemelos no ganan, se avisa: para esa opción hay que decir en la tarjeta y en `PUNTO5.md` que "los gemelos son mayormente condados vecinos".

In [20]:
if HAY_DATOS and KM is not None:
    def quienes_hacen(cultivo, opcion):
        if opcion == "nuevo":
            return SIEMBRAN.get(cultivo, set())
        return {f for f in perfil.index if hace_la_opcion(cultivo, opcion, f)}

    evaluar = rango[(rango["rotacion"] != "actual") & ~rango["condado_ya_lo_hace"].astype(bool) & (rango["n_gemelos"] >= MIN_GEMELOS)]
    filas = []
    for (cultivo, opcion), grupo in evaluar.groupby(["cultivo", "rotacion"]):
        hacen = quienes_hacen(cultivo, opcion)
        error_g, error_m, km_g, km_m = [], [], [], []
        for fips in grupo["fips"]:
            suyos = gemelos.loc[(gemelos["fips"] == fips) & (gemelos["cultivo"] == cultivo) & (gemelos["rotacion"] == opcion), "fips_gemelo"].tolist()
            mapa = cercanos_en_mapa(fips, hacen, len(suyos))
            error_g.append(abs(rinde.get(fips, np.nan) - rinde.reindex(suyos).mean()))
            error_m.append(abs(rinde.get(fips, np.nan) - rinde.reindex(mapa).mean()))
            km_g.append(KM.loc[fips, suyos].median())
            km_m.append(KM.loc[fips, mapa].median())
        filas.append({"cultivo": cultivo, "rotacion": opcion, "id_front": OPCION_FRONT[(cultivo, opcion)], "condados": len(grupo),
                      "error_gemelos": np.nanmean(error_g), "error_mapa": np.nanmean(error_m),
                      "km_gemelos": np.median(km_g), "km_mapa": np.median(km_m)})
    control_opcion = pd.DataFrame(filas)
    control_opcion["ganan_los_gemelos"] = control_opcion["error_gemelos"] < control_opcion["error_mapa"]
    print(f"Diferencia media de rinde de maíz (bu/ac), condados que NO hacen la opción | lluvia: {LLUVIA.upper()}")
    display(control_opcion.round(1))

    for _, f in control_opcion[~control_opcion["ganan_los_gemelos"]].iterrows():
        print(f"ATENCIÓN: en {f['id_front']} los gemelos no le ganan a los vecinos del mapa. "
              "Decir en la tarjeta que son mayormente condados vecinos.")
    if control_opcion["ganan_los_gemelos"].all():
        print("Los gemelos le ganan a los vecinos del mapa en todas las opciones.")
elif HAY_DATOS:
    control_opcion = pd.DataFrame()
    print("Sin la entrega no hay centroides: el control queda en espera.")
else:
    print("En espera de la tabla del punto 2.")

Diferencia media de rinde de maíz (bu/ac), condados que NO hacen la opción | lluvia: IMERG


,cultivo,rotacion,id_front,condados,error_gemelos,error_mapa,km_gemelos,km_mapa,ganan_los_gemelos
0,avena,nuevo,oats,451,12.7,13.8,222.0,155.0,True
1,maiz,continuo,corn_after_corn,547,20.3,21.5,205.3,129.2,True
2,maiz,rotado,corn_after_soy,483,16.0,22.6,178.0,108.4,True
3,soja,continuo,soy_after_soy,361,19.6,27.8,243.9,117.7,True
4,soja,rotado,soy_after_corn,448,20.7,23.9,208.0,130.3,True
5,trigo_invierno,nuevo,wheat,296,13.9,17.2,159.2,95.4,True


Los gemelos le ganan a los vecinos del mapa en todas las opciones.


## 20. Guardar

| Archivo | Qué tiene | Para quién |
|---|---|---|
| `backend/Datos/gemelos.csv` | Una fila por condado × cultivo × opción × gemelo: `fips_gemelo`, `distancia`, `similar_on`, `pct_sobre_si_mismo`, `pct_area`, `rend_tipico_bu_ac` | Revisión del equipo |
| `backend/Datos/gemelos_rango.csv` | Una fila por condado × cultivo × opción: `n_gemelos`, `n_anios`, `p10`, `mediana`, `p90`, `confianza`, `motivo`, `usar_como_rango` | Revisión del equipo |
| `backend/Datos/gemelos_control.csv` | Los errores de los controles de los pasos 9 y 19 | `PUNTO5.md` |
| `backend/Datos/front/gemelos.json` | Lo que lee el front, con los tipos de `web/src/next/types.ts` | El script que arma `next/<fips>.json` |

**Forma de `gemelos.json`** (`version_contrato` 0.2):

- `options`: el cruce de cada id del front con el punto 2. Por ejemplo, `corn_after_corn` = maíz `continuo`.
- `counties[fips]`: `fips`, `name`, `state` (sigla), `centroid` y:
  - `twins[]`: **un `Twin` del front por gemelo** (sin repetir), listo para copiar a `twins[]` de `next/<fips>.json`: `fips`, `name`, `state`, `centroid` como `[lon, lat]`, `similar_on` como lista, `grew_crop_pct` por cultivo (`{"corn": 48, "soy": 41}`, % del área sembrada), `source` y `prototype`.
  - `options[id]`: para cada opción, `crop`, `after` (cultivo anterior), `county_does_it`, `twins` (los fips, para `options[].twins` del front), `twins_range` (`{p10, median, p90}` o `null`), `use_as_range`, `n_years`, `confidence` (`high`, `medium` o `low`, con su motivo) y `text`.
- `text` solo muestra el rango cuando `use_as_range` es `true`. Si hay menos de 3 gemelos o los gemelos apuntan al revés que el modelo, lo dice en vez de dar un rango.

In [21]:
if HAY_DATOS:
    gemelos.to_csv(SALIDA / "gemelos.csv", index=False)
    rango.to_csv(SALIDA / "gemelos_rango.csv", index=False)

    filas_control = [{"control": "paso 9: todos los condados", "id_front": "", "metodo": metodo, "error_bu_ac": round(error, 2)}
                     for metodo, error in control_todos.items()]
    for _, f in control_opcion.iterrows():
        filas_control += [
            {"control": "paso 19: hacen la opción", "id_front": f["id_front"], "metodo": "Gemelos", "error_bu_ac": round(f["error_gemelos"], 2)},
            {"control": "paso 19: hacen la opción", "id_front": f["id_front"], "metodo": "Los más cercanos en el mapa que la hacen",
             "error_bu_ac": round(f["error_mapa"], 2)},
        ]
    pd.DataFrame(filas_control).assign(lluvia=LLUVIA).to_csv(SALIDA / "gemelos_control.csv", index=False)

    FUENTE = "USDA NASS · NASA IMERG · NASA POWER · SSURGO" if LLUVIA == "imerg" else "USDA NASS · NASA POWER · SSURGO"

    def numero(v):
        return None if pd.isna(v) else round(float(v), 1)

    def centroide(f):
        """[lon, lat], como lo espera el front (types.ts: centroid: [number, number])."""
        if f not in CONDADOS.index:
            return None
        return [round(float(CONDADOS.at[f, "lon_centroide"]), 4), round(float(CONDADOS.at[f, "lat_centroide"]), 4)]

    def area_por_cultivo(f):
        """grew_crop_pct del front: % del área sembrada de cada cultivo, con los ids de web/src/ui/crops.ts.
        Enteros desde 1 %; por debajo, con un decimal (la avena suele ocupar menos del 1 %). Los 0 no se mandan."""
        if f not in PARTE_AREA.index:
            return {}
        partes = {CULTIVO_FRONT[c]: (round(float(v)) if v >= 1 else round(float(v), 1)) for c, v in PARTE_AREA.loc[f].items() if pd.notna(v)}
        return {c: v for c, v in partes.items() if v > 0}

    def armar_gemelo(g):
        f = g["fips_gemelo"]
        return {"fips": f, "name": CONDADOS["condado"].get(f), "state": SIGLAS_ESTADO.get(f[:2]), "centroid": centroide(f),
                "similar_on": g["similar_on"].split("; "), "grew_crop_pct": area_por_cultivo(f),
                "source": FUENTE, "prototype": False}

    # Opciones con rango del modelo del punto 2 (fips, cultivo, rotación): fuera de esos estados no hay
    CON_MODELO = (set(zip(*[pd.read_csv(RUTA_MODELO_P2, dtype={"fips": str})[c] for c in ["fips", "cultivo", "rotacion"]]))
                  if RUTA_MODELO_P2.exists() else set())

    def armar_texto(r):
        quienes = OPCION_EN[(r["cultivo"], r["rotacion"])]
        aviso = "Similar soil and weather doesn't guarantee the same result."
        if r["n_gemelos"] == 0:
            return None
        if r["usar_como_rango"]:
            return (f"{quienes}: {r['n_gemelos']}. Their {CULTIVO_EN[r['cultivo']]} yield: {r['p10']:.0f}–{r['p90']:.0f} bu/ac "
                    f"(typical {r['mediana']:.0f}). {aviso}")
        if r["n_gemelos"] < MIN_GEMELOS:
            return f"{quienes}: {r['n_gemelos']}. Too few for a yield range. {aviso}"
        if (r["fips"], r["cultivo"], r["rotacion"]) in CON_MODELO:
            return f"{quienes}: {r['n_gemelos']}. Their yield mixes this rotation with other differences, so the range comes from the model. {aviso}"
        # Fuera de la zona maíz-soja el punto 2 no compara rotaciones a propósito: no es "todavía"
        return (f"{quienes}: {r['n_gemelos']}. Their yield mixes this rotation with other differences, "
                f"and the model doesn't compare rotations in this county. {aviso}")

    condados = {}
    for fips, de_fips in rango.groupby("fips"):
        suyos = gemelos[gemelos["fips"] == fips]
        opciones = {}
        for _, r in de_fips.iterrows():
            ids = suyos.loc[(suyos["cultivo"] == r["cultivo"]) & (suyos["rotacion"] == r["rotacion"]), "fips_gemelo"].tolist()
            opciones[OPCION_FRONT[(r["cultivo"], r["rotacion"])]] = {
                "crop": CULTIVO_FRONT[r["cultivo"]],
                "after": ANTECESOR_FRONT.get((r["cultivo"], r["rotacion"])),
                "county_does_it": bool(r["condado_ya_lo_hace"]),
                "twins": ids,
                "twins_range": ({"p10": numero(r["p10"]), "median": numero(r["mediana"]), "p90": numero(r["p90"])}
                                if r["usar_como_rango"] else None),
                "use_as_range": bool(r["usar_como_rango"]),
                "n_years": int(r["n_anios"]),
                "confidence": {"level": r["confianza"], "reason": r["motivo"]},
                "text": armar_texto(r),
            }
        condados[fips] = {
            "fips": fips, "name": CONDADOS["condado"].get(fips), "state": SIGLAS_ESTADO.get(fips[:2]), "centroid": centroide(fips),
            "twins": [armar_gemelo(g) for _, g in suyos.drop_duplicates("fips_gemelo").iterrows()],
            "options": opciones,
        }

    usadas = set(zip(rango["cultivo"], rango["rotacion"]))
    documento = {
        "version_contrato": "0.2", "generado": datetime.now().isoformat(timespec="seconds"),
        "es_prueba": False, "prototype": False, "metric": "yield_bu_ac", "tech_year": ANIO_TENDENCIA, "source": FUENTE,
        "note": ("Counties with similar soil and weather (2009–2023). Yield is for the whole county, not a single field. "
                 "When use_as_range is false, show the twins as who does it, and take the option's range from the model."),
        "options": {id_front: {"cultivo": c, "rotacion": o, "crop": CULTIVO_FRONT[c], "after": ANTECESOR_FRONT.get((c, o))}
                    for (c, o), id_front in OPCION_FRONT.items() if (c, o) in usadas},
        "counties": condados,
    }
    (SALIDA / "front").mkdir(exist_ok=True)
    with open(SALIDA / "front" / "gemelos.json", "w", encoding="utf-8") as f:
        json.dump(documento, f, ensure_ascii=False, indent=1)
    print(f"Guardados gemelos.csv ({len(gemelos)} filas), gemelos_rango.csv ({len(rango)} filas), "
          f"gemelos_control.csv y front/gemelos.json ({len(condados)} condados)")
else:
    print("En espera de la tabla del punto 2.")

Guardados gemelos.csv (30193 filas), gemelos_rango.csv (7160 filas), gemelos_control.csv y front/gemelos.json (895 condados)


## Próximos pasos

- [ ] Que el script que arma `web/public/data/next/<fips>.json` lea `backend/Datos/front/gemelos.json`: `counties[fips].twins` va directo a `twins[]`, y `options[id].twins` a `options[].twins`. El cruce de ids está en `options` del JSON (por ejemplo, `corn_after_corn` = maíz `continuo` del punto 2).
- [ ] Avena: `web/src/ui/crops.ts` no tiene id propio, así que va como `other` ("Other crops"). Si el front quiere mostrar "Oats", sumar `oats` a `crops.ts`.
- [ ] Decidir con el equipo qué mostrar cuando `use_as_range = false` (maíz rotado y continuo, porque los gemelos apuntan al revés que el modelo): hoy se muestran los gemelos como "quiénes lo hacen" y el rango sale del punto 2.
- [ ] Kansas (Sumner, el condado de demo del front) no está en la entrega: sin gemelos hasta que llegue.
- [ ] Probar más variables de clima con IMERG (por ejemplo, lluvia de julio) y repetir los controles de los pasos 9 y 19.
- [ ] Acordar con el equipo las variables y el peso del parecido (decisión abierta en `docs/front-end-definido.md`).